# Đánh Giá Sai Số & Chất Lượng Đầu-Cuối: KR260 (INT8 P1.5) vs. PyTorch FP32 / BF16 (20 WAVs)

Notebook này đối chiếu trực tiếp kết quả suy luận thực tế trên bo mạch **Xilinx Kria KR260 (`Mode 2: PS + PL INT8`)** với mô hình chuẩn **PyTorch `FP32` (Golden Baseline)** và **`BF16`** trên GPU cho toàn bộ **20 file `.wav`** (`01.wav` – `14.wav` gốc và `15_voiceassistant.wav` – `20_alpacaaudio.wav` Out-of-Distribution) được lưu trên Google Drive (`<WAV_DIR>`).

### Các chỉ số đánh giá chính:
1. **Semantic Similarity (`Cosine %`):** Độ tương đồng ngữ nghĩa câu trả lời bằng mô hình embedding `all-MiniLM-L6-v2`.
2. **ROUGE-L (`F1 %`) & Word Similarity (`WordSim %`):** Độ khớp cấu trúc từ vựng giữa câu trả lời của mạch FPGA INT8 và GPU FP32.
3. **Teacher-Forced Token Agreement (`Top-1 %`, `Top-5 %`):** Đánh giá độ khớp dự đoán ở mức logit trên từng bước sinh token của chuỗi câu trả lời (`1,451` tokens).

## 1. Cài Đặt Thư Viện (`!pip`)
Cài đặt các gói thư viện mô hình Qwen2-Audio, xử lý âm thanh và đo lường ngữ nghĩa (`sentence-transformers`, `rouge-score`).

In [1]:
!pip install -q --disable-pip-version-check transformers==4.55.0 accelerate==1.10.1 huggingface-hub==0.34.4 librosa soundfile sentence-transformers rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.


## 2. Kết Nối Google Drive & Kiểm Tra Thư Mục Dự Án
Kết nối Google Drive tại `/content/drive` và trỏ trực tiếp tới thư mục dự án `<PROJECT_DIR>/` (chứa `LLM_Question`, `pkg_v1` và `ps_pl_compare`).

In [2]:
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "<PROJECT_DIR>"
DRIVE_WAV_DIR = os.path.join(DRIVE_PROJECT_DIR, "LLM_Question")
DRIVE_PKG_DIR = os.path.join(DRIVE_PROJECT_DIR, "pkg_v1")
DRIVE_PS_PL_DIR = os.path.join(DRIVE_PROJECT_DIR, "ps_pl_compare")
OUT_DIR = os.path.join(DRIVE_PROJECT_DIR, "kr260_vs_fp32_out")
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Project Root   : {DRIVE_PROJECT_DIR} (exists={os.path.isdir(DRIVE_PROJECT_DIR)})")
print(f"WAV Folder     : {DRIVE_WAV_DIR} (exists={os.path.isdir(DRIVE_WAV_DIR)})")
print(f"PKG Folder     : {DRIVE_PKG_DIR} (exists={os.path.isdir(DRIVE_PKG_DIR)})")
print(f"PS/PL Compare  : {DRIVE_PS_PL_DIR} (exists={os.path.isdir(DRIVE_PS_PL_DIR)})")
print(f"Output Folder  : {OUT_DIR}")

Mounted at /content/drive
Project Root   : <PROJECT_DIR> (exists=True)
WAV Folder     : <WAV_DIR> (exists=True)
PKG Folder     : <PKG_DIR> (exists=True)
PS/PL Compare  : <PS_PL_COMPARE_DIR> (exists=True)
Output Folder  : <OUT_DIR>


## 3. Import Thư Viện & Kiểm Tra Phần Cứng GPU
Nạp toàn bộ các thư viện cần thiết và kiểm tra thiết bị CUDA.

In [3]:
import difflib
import gc
import glob
import json
import re
import subprocess
import sys
import time
import wave

import librosa
import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F
import transformers
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Device: {DEVICE}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)")

PyTorch: 2.14.1+cu130 | Transformers: 4.55.0 | Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GB VRAM)


## 4. Cấu Hình Link Model, Repo & Đọc Kết Quả Thực Tế Từ `ps_pl_compare`
Khai báo link Hugging Face checkpoint (`tranquangchung/qwen2-audio-dialogue`), GitHub repo (`Audio_Speech_Conversation`) và đọc kết quả 20 file `.wav` từ thư mục `<PS_PL_COMPARE_DIR>/ps_vs_pl_report.json`.

In [4]:
MODEL_ID = "tranquangchung/qwen2-audio-dialogue"
REPO_URL = "https://github.com/tranquangchung/Audio_Speech_Conversation"
REPO_DIR = "/content/Audio_Speech_Conversation"

DEFAULT_INSTRUCTION = (
    "You are a virtual assistant, answering questions and providing information "
    "in a clear and polite manner."
)
MAX_NEW_TOKENS = 200
TARGET_SR = 16000

# 1. Clone repo để lấy định nghĩa kiến trúc model (models/modeling_qwen2_audio.py & processing_qwen2_audio.py)
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 2. Đọc dữ liệu kết quả phần cứng KR260 từ thư mục ps_pl_compare trên Google Drive
REPORT_PATH = os.path.join(DRIVE_PS_PL_DIR, "pspl_compare_report.json")
if not os.path.isfile(REPORT_PATH):
    # Tự động tìm file .json trong ps_pl_compare nếu tên file khác
    json_cands = sorted(glob.glob(os.path.join(DRIVE_PS_PL_DIR, "*.json")))
    if json_cands:
        REPORT_PATH = json_cands[0]

assert os.path.isfile(REPORT_PATH), f"Không tìm thấy file report.json trong {DRIVE_PS_PL_DIR}"

with open(REPORT_PATH, "r", encoding="utf-8") as f:
    raw_rep = json.load(f)

KR260_DATA = [
    {
        "wav": r["wav"],
        "question": r["question"],
        "kr260_text": r["m2"]["text"],
        "kr260_tokens": r["m2"]["tokens"],
        "kr260_total_s": r["m2"]["total_s"],
        "kr260_tok_per_s": r["m2"]["tok_per_s"],
        "ps_total_s": r["m1"]["total_s"],
        "speedup_vs_ps": round(r["speedup"], 2),
        "ps_pl_word_sim": r["word_sim_pct"],
        "ps_pl_wer": r["wer_pct"],
    }
    for r in raw_rep
]

print(f"[OK] Loaded {len(KR260_DATA)} WAV records from: {REPORT_PATH}")
for idx, item in enumerate(KR260_DATA, 1):
    print(
        f"  [{idx:02d}/20] {item['wav']:<22} | "
        f"KR260 PL: {item['kr260_tokens']:3d} tok in {item['kr260_total_s']:5.2f}s "
        f"({item['speedup_vs_ps']:5.2f}x vs PS)"
    )

[OK] Loaded 20 WAV records from: <PS_PL_COMPARE_DIR>/pspl_compare_report.json
  [01/20] 01.wav                 | KR260 PL:  86 tok in 35.21s (10.80x vs PS)
  [02/20] 02.wav                 | KR260 PL:  98 tok in 35.12s (11.79x vs PS)
  [03/20] 03.wav                 | KR260 PL:  64 tok in 22.29s (11.91x vs PS)
  [04/20] 04.wav                 | KR260 PL:  69 tok in 26.25s (12.94x vs PS)
  [05/20] 05.wav                 | KR260 PL:  63 tok in 21.84s (11.85x vs PS)
  [06/20] 06.wav                 | KR260 PL: 134 tok in 41.88s ( 9.79x vs PS)
  [07/20] 07.wav                 | KR260 PL:  67 tok in 30.00s (14.33x vs PS)
  [08/20] 08.wav                 | KR260 PL:  76 tok in 30.12s (13.18x vs PS)
  [09/20] 09.wav                 | KR260 PL:  75 tok in 27.38s (12.05x vs PS)
  [10/20] 10.wav                 | KR260 PL:  96 tok in 34.27s (11.74x vs PS)
  [11/20] 11.wav                 | KR260 PL:  10 tok in  8.34s (20.13x vs PS)
  [12/20] 12.wav                 | KR260 PL:  59 tok in 30.08s (

## 5. Đọc & Kiểm Định 20 File `.wav` Từ Google Drive (`<WAV_DIR>`)
Đọc danh sách 20 file `.wav` trực tiếp từ Google Drive và xác thực định dạng chuẩn `RIFF/WAVE` (`16 kHz, 1-ch, 16-bit PCM`).

In [6]:
def verify_wav_file(path: str, expected_sr: int = 16000):
    if not os.path.isfile(path) or os.path.getsize(path) < 44:
        return False, "Missing or < 44 bytes"
    with open(path, "rb") as f:
        hdr = f.read(12)
    if hdr[:4] != b"RIFF" or hdr[8:12] != b"WAVE":
        return False, f"Invalid RIFF/WAVE header: {hdr!r}"
    try:
        info = sf.info(path)
        ok = (info.format == "WAV" and info.frames > 0)
        return ok, f"{info.samplerate}Hz | {info.channels}ch | {info.subtype} | {info.duration:.2f}s"
    except Exception as e:
        return False, f"sf.info error: {e}"


wav_folder = DRIVE_WAV_DIR
if not os.path.isdir(wav_folder):
    alt_folder = os.path.join(DRIVE_PROJECT_DIR, "LLM_Questions")
    if os.path.isdir(alt_folder):
        wav_folder = alt_folder

ALL_WAV_PATHS = sorted(glob.glob(os.path.join(wav_folder, "*.wav")))
assert len(ALL_WAV_PATHS) == 20, (
    f"Expected 20 .wav files in {wav_folder}, but found {len(ALL_WAV_PATHS)}. "
    f"Please check {DRIVE_WAV_DIR}"
)

print(f"[OK] Found and verified {len(ALL_WAV_PATHS)}/20 WAV files in {wav_folder}:")
for idx, wp in enumerate(ALL_WAV_PATHS, 1):
    ok, info = verify_wav_file(wp, TARGET_SR)
    status = "OK" if ok else "WARN"
    print(f"  [{idx:02d}/20] [{status}] {os.path.basename(wp):<22} | {info}")

[OK] Found and verified 20/20 WAV files in <WAV_DIR>:
  [01/20] [OK] 01.wav                 | 24000Hz | 1ch | FLOAT | 6.88s
  [02/20] [OK] 02.wav                 | 24000Hz | 1ch | FLOAT | 7.44s
  [03/20] [OK] 03.wav                 | 24000Hz | 1ch | FLOAT | 4.64s
  [04/20] [OK] 04.wav                 | 24000Hz | 1ch | FLOAT | 6.24s
  [05/20] [OK] 05.wav                 | 24000Hz | 1ch | FLOAT | 4.52s
  [06/20] [OK] 06.wav                 | 24000Hz | 1ch | FLOAT | 6.84s
  [07/20] [OK] 07.wav                 | 24000Hz | 1ch | FLOAT | 8.28s
  [08/20] [OK] 08.wav                 | 24000Hz | 1ch | FLOAT | 7.40s
  [09/20] [OK] 09.wav                 | 24000Hz | 1ch | FLOAT | 5.88s
  [10/20] [OK] 10.wav                 | 24000Hz | 1ch | FLOAT | 7.20s
  [11/20] [OK] 11.wav                 | 24000Hz | 1ch | FLOAT | 3.28s
  [12/20] [OK] 12.wav                 | 24000Hz | 1ch | FLOAT | 8.92s
  [13/20] [OK] 13.wav                 | 24000Hz | 1ch | FLOAT | 5.08s
  [14/20] [OK] 14.wav               

## 6. Nạp Mô Hình Qwen2-Audio (`FP32` Golden Baseline)
Tải processor và mô hình `tranquangchung/qwen2-audio-dialogue` từ Hugging Face lên GPU ở độ chính xác `FP32`.

In [7]:
from models.modeling_qwen2_audio import Qwen2AudioForConditionalGeneration
from models.processing_qwen2_audio import Qwen2AudioProcessor

t0 = time.perf_counter()
processor = Qwen2AudioProcessor.from_pretrained(MODEL_ID)
model_fp32 = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32, attn_implementation="eager"
).to(DEVICE).eval()
IM_END_ID = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
print(f"[OK] Loaded FP32 model on {DEVICE} in {time.perf_counter() - t0:.2f}s")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

... (8 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

[OK] Loaded FP32 model on cuda in 15.03s


## 7. Chạy Suy Luận `FP32` & `BF16` Trên Toàn Bộ 20 File `.wav`
Thực hiện sinh câu trả lời tự hồi quy (`max_new_tokens=200`, `do_sample=False`, `repetition_penalty=1.1`) ở cả chế độ `FP32` và `BF16` để đối chiếu với kết quả phần cứng KR260.

In [8]:
def build_inputs(wav_path, dtype=torch.float32):
    audio, _ = librosa.load(wav_path, sr=processor.feature_extractor.sampling_rate)
    prompt = (
        f"<|audio_bos|><|AUDIO|><|audio_eos|>"
        f"<|im_start|>user\n"
        f"Audio content provided. {DEFAULT_INSTRUCTION}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = processor(
        text=prompt,
        audio=audio,
        return_tensors="pt",
        sampling_rate=processor.feature_extractor.sampling_rate,
        padding=True,
    )
    return {
        k: (v.to(device=DEVICE, dtype=dtype) if v.is_floating_point() else v.to(DEVICE))
        for k, v in inputs.items()
    }


@torch.no_grad()
def run_generation_suite(model, dtype, label):
    results = []
    model.to(dtype=dtype)
    for idx, wp in enumerate(ALL_WAV_PATHS):
        wname = os.path.basename(wp)
        inputs = build_inputs(wp, dtype=dtype)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=IM_END_ID,
            pad_token_id=processor.tokenizer.pad_token_id,
            use_cache=True,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        in_len = inputs["input_ids"].size(1)
        out_ids = gen_ids[:, in_len:]
        n_tok = int(out_ids.size(1))
        text = processor.batch_decode(out_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        text = " ".join(text.strip().split())
        print(f"[{label}] [{idx+1:02d}/20] {wname:<22} | {n_tok:3d} tokens | {dt:.2f}s ({n_tok/max(dt,1e-6):.1f} tok/s)")
        results.append({
            "wav": wname,
            "tokens": n_tok,
            "time_s": round(dt, 3),
            "tok_per_s": round(n_tok / max(dt, 1e-6), 2),
            "text": text,
            "out_ids": out_ids[0].detach().cpu(),
        })
    return results


print("=== Running FP32 Golden Generation on 20 WAVs ===")
fp32_gen = run_generation_suite(model_fp32, torch.float32, "FP32")

print("\n=== Running BF16 Generation on 20 WAVs ===")
bf16_gen = run_generation_suite(model_fp32, torch.bfloat16, "BF16")

# Khôi phục model về FP32 để đo Teacher-Forced Logits ở bước tiếp theo
model_fp32.to(dtype=torch.float32)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

=== Running FP32 Golden Generation on 20 WAVs ===
[FP32] [01/20] 01.wav                 |  91 tokens | 5.15s (17.7 tok/s)
[FP32] [02/20] 02.wav                 |  90 tokens | 0.99s (91.1 tok/s)
[FP32] [03/20] 03.wav                 |  85 tokens | 0.94s (90.6 tok/s)
[FP32] [04/20] 04.wav                 | 100 tokens | 1.08s (92.7 tok/s)
[FP32] [05/20] 05.wav                 |  57 tokens | 0.66s (86.7 tok/s)
[FP32] [06/20] 06.wav                 |  99 tokens | 1.08s (91.8 tok/s)
[FP32] [07/20] 07.wav                 |  77 tokens | 0.86s (89.7 tok/s)
[FP32] [08/20] 08.wav                 |  74 tokens | 0.83s (89.1 tok/s)
[FP32] [09/20] 09.wav                 |  62 tokens | 0.71s (87.7 tok/s)
[FP32] [10/20] 10.wav                 | 112 tokens | 1.21s (92.5 tok/s)
[FP32] [11/20] 11.wav                 |  10 tokens | 0.19s (52.5 tok/s)
[FP32] [12/20] 12.wav                 |  38 tokens | 0.47s (80.3 tok/s)
[FP32] [13/20] 13.wav                 |  24 tokens | 0.33s (72.4 tok/s)
[FP32] [14/20]

## 8. Đo Sai Số Logit & Token Agreement (Teacher-Forced Theo Chuỗi KR260 INT8 vs. FP32)
Đưa chuỗi token của câu trả lời KR260 INT8 qua mô hình `FP32` (Teacher-Forced) để đo tỷ lệ khớp dự đoán **Top-1 (%)** và **Top-5 (%)** trên từng bước sinh token.

In [9]:
def apply_rep_penalty(logits_1d, history_ids, penalty=1.1):
    logits_p = logits_1d.clone()
    uniq = torch.unique(history_ids)
    vals = logits_p[uniq]
    logits_p[uniq] = torch.where(vals > 0, vals / penalty, vals * penalty)
    return logits_p


@torch.no_grad()
def evaluate_teacher_forced_agreement(wav_path, kr260_text):
    inputs = build_inputs(wav_path, dtype=torch.float32)
    prompt_ids = inputs["input_ids"][0]

    kr_ids = processor.tokenizer.encode(kr260_text, add_special_tokens=False)
    kr_ids_t = torch.tensor(kr_ids, dtype=torch.long, device=DEVICE)
    T_gen = kr_ids_t.numel()
    if T_gen == 0:
        return {"top1_pct": 100.0, "top5_pct": 100.0, "top1_hits": 0, "top5_hits": 0, "n_tokens": 0}

    full_ids = torch.cat([prompt_ids, kr_ids_t[:-1]], dim=0).unsqueeze(0)
    full_mask = torch.ones_like(full_ids)
    fwd_inputs = dict(inputs)
    fwd_inputs["input_ids"] = full_ids
    fwd_inputs["attention_mask"] = full_mask

    out = model_fp32(**fwd_inputs)
    step_logits = out.logits[0, -T_gen:, :].float()

    top1_hits = 0
    top5_hits = 0
    seq_so_far = prompt_ids.clone()
    for t in range(T_gen):
        l_row = apply_rep_penalty(step_logits[t], seq_so_far, penalty=1.1)
        top5 = torch.topk(l_row, k=5).indices
        target_tok = kr_ids_t[t]
        if top5[0] == target_tok:
            top1_hits += 1
        if (top5 == target_tok).any():
            top5_hits += 1
        seq_so_far = torch.cat([seq_so_far, target_tok.view(1)], dim=0)

    return {
        "top1_pct": 100.0 * top1_hits / T_gen,
        "top5_pct": 100.0 * top5_hits / T_gen,
        "top1_hits": top1_hits,
        "top5_hits": top5_hits,
        "n_tokens": int(T_gen),
    }


tf_metrics = []
for idx, (wp, kr) in enumerate(zip(ALL_WAV_PATHS, KR260_DATA)):
    m = evaluate_teacher_forced_agreement(wp, kr["kr260_text"])
    tf_metrics.append(m)
    print(
        f"[TF Check] [{idx+1:02d}/20] {kr['wav']:<22} | "
        f"Top-1 Agreement: {m['top1_pct']:6.2f}% ({m['top1_hits']}/{m['n_tokens']}) | "
        f"Top-5 Agreement: {m['top5_pct']:6.2f}%"
    )

[TF Check] [01/20] 01.wav                 | Top-1 Agreement:  87.06% (74/85) | Top-5 Agreement: 100.00%
[TF Check] [02/20] 02.wav                 | Top-1 Agreement:  93.81% (91/97) | Top-5 Agreement: 100.00%
[TF Check] [03/20] 03.wav                 | Top-1 Agreement:  93.65% (59/63) | Top-5 Agreement: 100.00%
[TF Check] [04/20] 04.wav                 | Top-1 Agreement:  89.71% (61/68) | Top-5 Agreement: 100.00%
[TF Check] [05/20] 05.wav                 | Top-1 Agreement:  95.16% (59/62) | Top-5 Agreement: 100.00%
[TF Check] [06/20] 06.wav                 | Top-1 Agreement:  88.64% (117/132) | Top-5 Agreement:  99.24%
[TF Check] [07/20] 07.wav                 | Top-1 Agreement:  90.91% (60/66) | Top-5 Agreement: 100.00%
[TF Check] [08/20] 08.wav                 | Top-1 Agreement:  89.33% (67/75) | Top-5 Agreement:  98.67%
[TF Check] [09/20] 09.wav                 | Top-1 Agreement:  93.24% (69/74) | Top-5 Agreement: 100.00%
[TF Check] [10/20] 10.wav                 | Top-1 Agreement:  

## 9. Tính Toán Độ Tương Đồng Ngữ Nghĩa (`Semantic Cosine`), `ROUGE-L` & `WordSim`
Sử dụng mô hình `SentenceTransformer("all-MiniLM-L6-v2")` và `ROUGE-L` để đo độ tương đồng về mặt ý nghĩa và cấu trúc từ vựng giữa câu trả lời của **KR260 FPGA INT8** và **GPU PyTorch FP32 / BF16**.

In [10]:
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))
r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def normalize_words(text):
    return re.findall(r"[a-z0-9']+", text.lower())


def calc_text_pair_metrics(ref_text, hyp_text):
    emb = embedder.encode([ref_text, hyp_text], convert_to_tensor=True, normalize_embeddings=True)
    sem_cos = 100.0 * float(torch.sum(emb[0] * emb[1]).item())
    rouge_l = 100.0 * r_scorer.score(ref_text, hyp_text)["rougeL"].fmeasure
    rw, hw = normalize_words(ref_text), normalize_words(hyp_text)
    word_sim = 100.0 * difflib.SequenceMatcher(None, rw, hw).ratio() if (rw or hw) else 100.0
    char_sim = 100.0 * difflib.SequenceMatcher(None, ref_text.lower(), hyp_text.lower()).ratio()
    return {
        "sem_cos_pct": round(sem_cos, 2),
        "rouge_l_pct": round(rouge_l, 2),
        "word_sim_pct": round(word_sim, 2),
        "char_sim_pct": round(char_sim, 2),
    }


comparison_rows = []
for i in range(20):
    kr = KR260_DATA[i]
    fp = fp32_gen[i]
    bf = bf16_gen[i]
    tf = tf_metrics[i]

    m_kr_vs_fp32 = calc_text_pair_metrics(fp["text"], kr["kr260_text"])
    m_bf_vs_fp32 = calc_text_pair_metrics(fp["text"], bf["text"])

    comparison_rows.append({
        "wav": kr["wav"],
        "group": "Original (01-14)" if i < 14 else "OOD New (15-20)",
        "question": kr["question"],
        "fp32_text": fp["text"],
        "bf16_text": bf["text"],
        "kr260_int8_text": kr["kr260_text"],
        "fp32_tokens": fp["tokens"],
        "kr260_tokens": kr["kr260_tokens"],
        "kr260_total_s": kr["kr260_total_s"],
        "kr260_speedup_vs_ps": kr["speedup_vs_ps"],
        "tf_top1_pct": round(tf["top1_pct"], 2),
        "tf_top5_pct": round(tf["top5_pct"], 2),
        "tf_top1_hits": tf["top1_hits"],
        "tf_n_tokens": tf["n_tokens"],
        "kr260_vs_fp32": m_kr_vs_fp32,
        "bf16_vs_fp32": m_bf_vs_fp32,
    })

print("[OK] Computed semantic and lexical similarity metrics for all 20 WAVs.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[OK] Computed semantic and lexical similarity metrics for all 20 WAVs.


## 10. Xuất Bảng Đối Sánh Toàn Diện (20 WAVs) & Lưu File Báo Cáo
In chi tiết câu trả lời `FP32` vs. `KR260 INT8`, bảng tổng hợp các chỉ số sai số và lưu ra file `fp32_vs_kr260_int8_report.json` trong thư mục `<OUT_DIR>`.

In [11]:
print("=" * 100)
print("PER-WAV QUALITATIVE & SEMANTIC COMPARISON: PYTORCH FP32 (GPU) vs. KR260 INT8 (PS + PL)")
print("=" * 100)
for row in comparison_rows:
    m = row["kr260_vs_fp32"]
    print("-----")
    print(f"WAV               : {row['wav']} [{row['group']}]")
    print(f"Question          : {row['question']}")
    print(f"Answer (FP32 GPU) : {row['fp32_text']}")
    print(f"Answer (KR260 PL) : {row['kr260_int8_text']}")
    print(
        f"Metrics vs. FP32  : Semantic Cosine = {m['sem_cos_pct']:.2f}% | "
        f"ROUGE-L = {m['rouge_l_pct']:.2f}% | WordSim = {m['word_sim_pct']:.2f}% | "
        f"TF Top-1 = {row['tf_top1_pct']:.2f}% | TF Top-5 = {row['tf_top5_pct']:.2f}%"
    )

print("\n" + "=" * 145)
print("QUANTITATIVE COMPARISON TABLE: KR260 INT8 (PS + PL) vs. PYTORCH FP32 GOLDEN BASELINE (20 WAVs)")
print("=" * 145)
hdr = (
    f"{'WAV':<22} | {'FP32 Tok':>8} {'KR260 Tok':>9} | "
    f"{'TF Top-1':>8} {'TF Top-5':>8} | "
    f"{'SemCos(INT8)':>12} {'SemCos(BF16)':>12} | "
    f"{'ROUGE-L':>8} {'WordSim':>8} {'CharSim':>8} | {'KR260 Time':>10}"
)
print(hdr)
print("-" * len(hdr))

for r in comparison_rows:
    m_kr = r["kr260_vs_fp32"]
    m_bf = r["bf16_vs_fp32"]
    print(
        f"{r['wav']:<22} | {r['fp32_tokens']:8d} {r['kr260_tokens']:9d} | "
        f"{r['tf_top1_pct']:7.2f}% {r['tf_top5_pct']:7.2f}% | "
        f"{m_kr['sem_cos_pct']:11.2f}% {m_bf['sem_cos_pct']:11.2f}% | "
        f"{m_kr['rouge_l_pct']:7.2f}% {m_kr['word_sim_pct']:7.2f}% {m_kr['char_sim_pct']:7.2f}% | "
        f"{r['kr260_total_s']:9.2f}s"
    )
print("-" * len(hdr))


def summarize_group(subset, name):
    n = len(subset)
    tot_hits1 = sum(r["tf_top1_hits"] for r in subset)
    tot_hits5 = sum(int(round(r["tf_top5_pct"] * r["tf_n_tokens"] / 100.0)) for r in subset)
    tot_tok = sum(r["tf_n_tokens"] for r in subset)
    w_top1 = 100.0 * tot_hits1 / max(tot_tok, 1)
    w_top5 = 100.0 * tot_hits5 / max(tot_tok, 1)
    avg_sem_kr = sum(r["kr260_vs_fp32"]["sem_cos_pct"] for r in subset) / n
    avg_sem_bf = sum(r["bf16_vs_fp32"]["sem_cos_pct"] for r in subset) / n
    avg_rouge = sum(r["kr260_vs_fp32"]["rouge_l_pct"] for r in subset) / n
    avg_wsim = sum(r["kr260_vs_fp32"]["word_sim_pct"] for r in subset) / n
    avg_csim = sum(r["kr260_vs_fp32"]["char_sim_pct"] for r in subset) / n
    print(f"\n[{name} (N={n}, Total KR260 Tokens={tot_tok})]")
    print(f"  Teacher-Forced Top-1 Token Agreement (INT8 vs FP32) : {w_top1:.2f}% ({tot_hits1}/{tot_tok})")
    print(f"  Teacher-Forced Top-5 Token Agreement (INT8 vs FP32) : {w_top5:.2f}%")
    print(f"  Avg Semantic Cosine Similarity (KR260 INT8 vs FP32) : {avg_sem_kr:.2f}% (vs BF16: {avg_sem_bf:.2f}%)")
    print(f"  Avg ROUGE-L F1 Score (KR260 INT8 vs FP32)           : {avg_rouge:.2f}%")
    print(f"  Avg Lexical Word / Char Similarity                  : {avg_wsim:.2f}% / {avg_csim:.2f}%")


summarize_group(comparison_rows, "ALL 20 WAV FILES")
summarize_group(comparison_rows[:14], "ORIGINAL 14 WAV FILES (01-14)")
summarize_group(comparison_rows[14:], "NEW 6 OOD WAV FILES (15-20)")

json_out = os.path.join(OUT_DIR, "fp32_vs_kr260_int8_report.json")
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, indent=2, ensure_ascii=False)
print(f"\n[OK] Saved detailed JSON report to: {json_out}")

PER-WAV QUALITATIVE & SEMANTIC COMPARISON: PYTORCH FP32 (GPU) vs. KR260 INT8 (PS + PL)
-----
WAV               : 01.wav [Original (01-14)]
Question          : If you had to live in a video game universe for a year, which one would you choose?
Answer (FP32 GPU) : That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were a good old-fashioned video game world. There wouldn't be any conflict or violence between players. On the other hand, some people might find that a more realistic or fair scenario could have more consequences. It really depends on what kind of balance you're looking for in such a setting. What about you? Which universe do you imagine being the most balanced?
Answer (KR260 PL) : That's an interesting question! Personally, I think the universe would be a very peaceful one. It wouldn't have as much conflict or violence as some other universes might have. On the other hand, if it were a chaotic universe with lots of different wo